## Imports para criar o RAG

In [15]:
import json #ler o arquivo json
import os #lista os arquivos do diretório

from langchain_classic.chains import RetrievalQA # orquestra o processo do RAG (Modelo + recuperador)\n
from langchain_ollama import ChatOllama # instanciando o modelo de linguagem (para testes)
from langchain_community.document_loaders import PyPDFLoader #carregar documentos PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter #cortador de texto (caracteres recursivos)\n
from langchain_community.vectorstores import FAISS #busca por similaridade (vetores)
from langchain_ollama import OllamaEmbeddings #instanciando o modelo de embeddings (vetores)

## Definir as variáveis de ambiente

In [16]:
with open ("../config/config.json", "r") as f:
    config = json.load(f)
    
llm = ChatOllama(model=config["MODEL"], temperature=0)
embeddings = OllamaEmbeddings(model=config["EMBEDDING_MODEL"])
DATA_DIR = config["DATA_DIR"]
VECTOR_STORE_DIR = config["VECTOR_STORE_DIR"]

## Carregar e cortar os PDFs

In [ ]:
def carregar_cortar_pdfs(data_dir):
    all_docs = []
    for filename in os.listdir(data_dir):
        if filename.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(data_dir, filename))
            docs = loader.load()
            all_docs.extend(docs)

    if not all_docs:
        raise ValueError("Sem PDFs na pasta de dados")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, #aprox 1k caractere por bloco
        chunk_overlap=200 #sobreposição em aprox 200 caracteres (continuidade)
        )
    chunks = splitter.split_documents(all_docs)
    return chunks

chunks = carregar_cortar_pdfs(f"../{DATA_DIR}")
print(f"carregado e cortado em {len(chunks)} chunks.")

carregado e cortado em 67 chunks.


## Criando o armazenamento de vetores

In [ ]:
def criar_vectorstore(chunks, vectorstoreDir):
    embeddings = OllamaEmbeddings(model=config["EMBEDDING_MODEL"]) #instanciando o modelo de embeddings (vetores)
    vectorstore = FAISS.from_documents(chunks, embeddings) # cria o vetorstore a partir dos chunks e do modelo de embeddings
    vectorstore.save_local(vectorstoreDir) # salva o vetorstore localmente
    
    print(f"Vectorstore criado e salvo em {vectorstoreDir}")

vectorstoreDir = f"../{VECTOR_STORE_DIR}"
criar_vectorstore(chunks, vectorstoreDir)

## Carregar o vectorstore
Agora iremos carregar o armazenamento de vetores criado no passo anterior

In [18]:
vector_dir = f"../{VECTOR_STORE_DIR}"

def carregar_vectorstore(vectorstoreDir):
    embeddings = OllamaEmbeddings(model=config["EMBEDDING_MODEL"]) #instanciando o modelo de embeddings (vetores)
    vectorstore = FAISS.load_local(
        folder_path=vectorstoreDir, 
        embeddings=embeddings,
        allow_dangerous_deserialization = True
        ) #carrega o vetorstore localmente
    return vectorstore

def perguntar_ao_rag(pergunta):
    vectorstore = carregar_vectorstore(vector_dir)
    retriever = vectorstore.as_retriever() #recuperador de documento
    llm = ChatOllama(model=config["MODEL"], temperature=0) #instanciando o modelo de linguagem (para testes)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever) #orquestra o processo do RAG (Modelo + recuperador)
    return qa.run(pergunta) #retorna a resposta do RAG

#pergunta de teste
perguntar_ao_rag("O que são rolamentos?")

'Rolamentos são componentes mecânicos que reduzem o atrito entre partes móveis de uma máquina, permitindo o movimento suave e eficiente. Eles são essenciais para o funcionamento de máquinas rotativas, suportando o peso e as cargas, enquanto minimizam o desgaste e as vibrações. Em condições normais, eles são inspecionados regularmente para garantir sua integridade e evitar problemas como desgaste, desalinhamento ou falhas que podem causar danos ou desfuncionamento.'

In [20]:
perguntar_ao_rag("um rolamento típico é composto por?")

'The provided context discusses inspection parameters for bearings, such as radial and axial play, temperature, and noise, but does not list the physical components of a bearing. The standard components of a bearing typically include:  \n1. **Inner Ring** (sealed or open)  \n2. **Outer Ring** (sealed or open)  \n3. **Ball/Particulate** (for ball bearings) or **Journal** (for journal bearings)  \n4. **Cage** (to hold the balls or particles)  \n5. **Seal** (to prevent contamination)  \n\nSince the context does not specify these components, the answer cannot be derived from the given information.'